# Satellite Change Detection — Walkthrough

A notebook walkthrough of the end-to-end pipeline:
1. Pick a region (bounding box)
2. Fetch land-cover maps from Planetary Computer
3. Compute change detection + statistics
4. Render maps, charts, and a report

In [ ]:
# Make sure the project root is importable
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.data import BBox, fetch_lulc, fetch_s2_rgb_preview
from src.change_detection import compute_change, format_change_report, top_transitions
from src.viz import render_lulc_map, render_change_map, render_rgb_preview, transition_bar_chart

## 1. Pick a region and years

Using Rondônia, Brazil — a well-known deforestation frontier.

In [ ]:
bbox = BBox(west=-63.2, south=-10.7, east=-62.5, north=-10.1)
before_year = 2018
after_year = 2023
print(f'AOI area ~{bbox.area_km2_approx():.0f} km²')

## 2. Fetch land cover for both years

In [ ]:
before = fetch_lulc(bbox, before_year).compute()
after = fetch_lulc(bbox, after_year).compute()
if after.shape != before.shape:
    after = after.interp_like(before, method='nearest').astype('int16')
print('before shape:', before.shape, 'after shape:', after.shape)

## 3. Visualize the before / after land cover

In [ ]:
fig = render_lulc_map(before, title=f'Land cover {before_year}')
fig.show()

In [ ]:
fig = render_lulc_map(after, title=f'Land cover {after_year}')
fig.show()

## 4. Compute change and inspect top transitions

In [ ]:
result = compute_change(before, after, before_year, after_year)
for fc, tc, ha in top_transitions(result, n=10):
    print(f'{fc:2d} -> {tc:2d}  {ha:,.0f} ha')

In [ ]:
fig = render_change_map(result)
fig.show()

In [ ]:
fig = transition_bar_chart(result, n=8)
fig.show()

## 5. Print the report

In [ ]:
print(format_change_report(result, aoi_name='Rondônia, Brazil'))

## 6. (Optional) Sentinel-2 RGB previews for context

In [ ]:
rgb_before = fetch_s2_rgb_preview(bbox, before_year)
if rgb_before is not None:
    rgb_before = rgb_before.compute()
    fig = render_rgb_preview(rgb_before, title=f'RGB {before_year}')
    fig.show()
else:
    print('No cloud-free imagery for this AOI/year')